In [ ]:
import mne
from pathlib import Path
import os
import numpy as np

SUBJECT = "16zolotov_matvei"
SESSION = "C"

USE_TEMPLATE = True

DO_TRIALWISE_ROI = True
ROI_PARC = "aparc.a2009s"

# ROI labels for timecourse extraction (Destrieux atlas, aparc.a2009s).
# None = all labels in the parcellation (for debugging / checking label names).
#
# vmPFC = ventromedial PFC:
#   G_rectus         — gyrus rectus (most medial/ventral)
#   G_and_S_frontomargin — frontomarginal gyrus/sulcus
#   G_orbital        — orbitofrontal (lateral OFC)
#   S_orbital_med-olfact — medial orbital/olfactory sulcus
#   S_orbital-H_Shaped   — H-shaped orbital sulcus
#
# dlPFC = dorsolateral PFC:
#   G_front_middle   — middle frontal gyrus (dlPFC core)
#   G_front_sup      — superior frontal gyrus (dorsal part)
#   S_front_sup/middle/inf — corresponding sulci
ROI_LABELS_OF_INTEREST = [
    # ── vmPFC ────────────────────────────────────────────────
    "G_rectus-lh",                  "G_rectus-rh",
    "G_and_S_frontomargin-lh",      "G_and_S_frontomargin-rh",
    "G_orbital-lh",                 "G_orbital-rh",
    "S_orbital_med-olfact-lh",      "S_orbital_med-olfact-rh",
    "S_orbital-H_Shaped-lh",        "S_orbital-H_Shaped-rh",
    # ── dlPFC ────────────────────────────────────────────────
    "G_front_middle-lh",            "G_front_middle-rh",
    "G_front_sup-lh",               "G_front_sup-rh",
    "S_front_sup-lh",               "S_front_sup-rh",
    "S_front_middle-lh",            "S_front_middle-rh",
    "S_front_inf-lh",               "S_front_inf-rh",
]

MEG_BASE = Path(r"E:\Develop\MEG")

RAW_FILE_PATH = MEG_BASE / SUBJECT / SESSION / "derivatives" / f"epochs_combined_{SESSION}-epo.fif"

DERIVATIVES_BASE = MEG_BASE / SUBJECT / SESSION / "derivatives"

if USE_TEMPLATE:
    SUBJECTS_DIR = Path(r"E:\Develop\MEG\freesurfer_subjects\fsaverage")
    freesurfer_dir = SUBJECTS_DIR  # ← freesurfer_dir = fsaverage itself
    TEMPLATE_NAME = "fsaverage"
    OUTPUT_DIR = DERIVATIVES_BASE / "fsaverage"
    FREESURFER_NAME = "fsaverage"

    bem_path = SUBJECTS_DIR / "bem" / f"{TEMPLATE_NAME}-5120-5120-5120-bem-sol.fif"
    src_path = SUBJECTS_DIR / "bem" / f"{TEMPLATE_NAME}-ico-5-src.fif"
else:
    SUBJECTS_DIR = MEG_BASE / SUBJECT
    TEMPLATE_NAME = SUBJECT
    OUTPUT_DIR = DERIVATIVES_BASE / "NativeMRI"
    bem_path = None
    src_path = None
    freesurfer_dir = None

    print(f"Searching for FreeSurfer directory in: {SUBJECTS_DIR}")
    for subdir in SUBJECTS_DIR.iterdir():
        if subdir.is_dir() and (subdir / "bem").exists():
            freesurfer_dir = subdir
            print(f"Found directory: {subdir.name}")
            bem_files = list(subdir.glob("bem/*-bem-sol.fif"))
            if bem_files:
                bem_path = bem_files[0]
                print(f"BEM file: {bem_path.name}")
            src_files = list(subdir.glob("bem/*-src.fif"))
            if src_files:
                src_path = src_files[0]
                print(f"SRC file: {src_path.name}")
            break

    if bem_path is None or src_path is None:
        print(f"ERROR: BEM or SRC files not found in {SUBJECTS_DIR}")
        if freesurfer_dir:
            print(f"Found directory: {freesurfer_dir.name}")
            print(f"BEM files: {list((freesurfer_dir / 'bem').glob('*-bem-sol.fif'))}")
            print(f"SRC files: {list((freesurfer_dir / 'bem').glob('*-src.fif'))}")
        import sys
        sys.exit(1)

    FREESURFER_NAME = freesurfer_dir.name

print(f"FREESURFER_NAME = {FREESURFER_NAME}")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ROI_TC_OUT = OUTPUT_DIR / f"roi_tc_{ROI_PARC}_{SESSION}.npz"

trans_out = OUTPUT_DIR / f"sub-{SUBJECT}_ses-{SESSION}_desc-{FREESURFER_NAME}_trans.fif"

inv_out = OUTPUT_DIR / f"sub-{SUBJECT}_ses-{SESSION}_desc-{FREESURFER_NAME}_inv.fif"
stc_avg_path = OUTPUT_DIR / f"sub-{SUBJECT}_ses-{SESSION}_desc-{FREESURFER_NAME}_avg-src"

def check_files(*paths):
    all_ok = True
    for p in paths:
        p = Path(p)
        if not p.exists():
            print(f"NOT FOUND: {p}")
            all_ok = False
        else:
            print(f"OK: {p}")
    return all_ok

def print_config():
    mode_str = "TEMPLATE (fsaverage)" if USE_TEMPLATE else "REAL MRI"
    print("\n" + "="*70)
    print(f"MODE:           {mode_str}")
    print(f"SUBJECT:        {SUBJECT}")
    print(f"SESSION:        {SESSION}")
    print(f"SUBJECTS_DIR:   {SUBJECTS_DIR}")
    if not USE_TEMPLATE:
        print(f"FREESURFER:     {FREESURFER_NAME}")
    print(f"RAW_FILE:       {RAW_FILE_PATH}")
    print(f"OUTPUT_DIR:     {OUTPUT_DIR}")
    print(f"  └─ TRANS:     {trans_out}")
    print(f"  └─ INV:       {inv_out}")
    print(f"  └─ STC:       {stc_avg_path}")
    print(f"BEM path:       {bem_path}")
    print(f"SRC path:       {src_path}")
    n_roi = len(ROI_LABELS_OF_INTEREST) if ROI_LABELS_OF_INTEREST else "ALL"
    print(f"ROI:            {n_roi} labels  ({ROI_PARC})")
    if ROI_LABELS_OF_INTEREST:
        vmpfc = [l for l in ROI_LABELS_OF_INTEREST if any(k in l for k in ("rectus","frontomargin","orbital"))]
        dlpfc = [l for l in ROI_LABELS_OF_INTEREST if any(k in l for k in ("front_middle","front_sup","front_inf"))]
        print(f"  └─ vmPFC:   {len(vmpfc)} labels")
        print(f"  └─ dlPFC:   {len(dlpfc)} labels")
    print("="*70 + "\n")

print_config()


In [ ]:
# ===========================================================
# ANATOMY ABSTRACTION LAYER
# ONE-LINE MODE SWITCH: change ANATOMY_MODE to "subject_mri"
# when individual FreeSurfer reconstructions are ready.
# ===========================================================
import sys
from pathlib import Path

_repo_root = Path(r"E:\Develop\MEG\Preprocessing_Pipeline")
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

from pipeline_utils.anatomy_layer import get_anatomy

ANATOMY_MODE = "fsaverage"   # "subject_mri" for individual MRI
anat = get_anatomy(SUBJECT, SESSION, mode=ANATOMY_MODE, meg_root=MEG_BASE)

bem_path        = anat.bem_path
src_path        = anat.src_path
OUTPUT_DIR      = anat.trans_dir
FREESURFER_NAME = anat.freesurfer_name

# ─────────────────────────────────────────────────────────────────────────────
# BIDS-style per-subject/per-session file names
# Format: sub-{SUBJECT}_ses-{SESSION}_desc-{mode}_{type}.fif
# Reviewers can identify WHO, WHICH SESSION, WHICH SPACE from the filename.
# ─────────────────────────────────────────────────────────────────────────────
_desc = ANATOMY_MODE  # "fsaverage" or "subjectMRI"
trans_out    = OUTPUT_DIR / f"sub-{SUBJECT}_ses-{SESSION}_desc-{_desc}_trans.fif"
inv_out      = OUTPUT_DIR / f"sub-{SUBJECT}_ses-{SESSION}_desc-{_desc}_inv.fif"
stc_avg_path = OUTPUT_DIR / f"sub-{SUBJECT}_ses-{SESSION}_desc-{_desc}_avg-src"
ROI_TC_OUT      = OUTPUT_DIR / f"roi_tc_{ROI_PARC}_{SESSION}.npz"
ROI_TC_LCMV_OUT = OUTPUT_DIR / f"roi_tc_LCMV_{SESSION}.npz"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"[anatomy] mode={ANATOMY_MODE}  subject={anat.freesurfer_name}")
print(f"  bem   : {anat.bem_path.name}")
print(f"  src   : {anat.src_path.name}")
print(f"  outdir: {anat.trans_dir}")
print(f"[paths — BIDS-style]")
print(f"  trans : {trans_out.name}")
print(f"  inv   : {inv_out.name}")
print(f"  fwd   : sub-{SUBJECT}_ses-{SESSION}_desc-{_desc}_fwd.fif  (saved in STEP 8.1)")
print(f"  cov   : sub-{SUBJECT}_ses-{SESSION}_desc-{{emptyroom|baseline}}_noise-cov.fif  (STEP 9)")


In [ ]:
bem_path_debug = bem_path
bem_debug = mne.read_bem_solution(str(bem_path_debug))
print(f"✓ BEM loaded from: {TEMPLATE_NAME}")
print(f"  Surfaces: {len(bem_debug['surfs'])}")
print(f"  First (skin): {bem_debug['surfs'][0]['rr'].shape[0]} vertices")



KeyboardInterrupt



In [ ]:
print(f"Checking required files...\n")

files_to_check = [
    RAW_FILE_PATH,
    bem_path,
]

all_ok = check_files(*files_to_check)

if not all_ok:
    print("\nWARNING: check paths in Cell 1 (config)!")
    raise FileNotFoundError("Some files missing")

print("\n✓ All required files found\n")


In [ ]:
# DIAGNOSTICS: digitizer quality

print(f"Digitizer quality analysis...\n")

if 'raw' not in locals():
    print("  Loading data...")
    epochs = mne.read_epochs(str(RAW_FILE_PATH), preload=False, verbose="ERROR")


all_digs_raw = np.array([d['r'] for d in epochs.info['dig'] if d['kind'] == 4])
print(f"DIGITIZER STATS:")
print(f"  Total points: {len(all_digs_raw)}")
print(f"  Mean dist from center: {np.linalg.norm(all_digs_raw - all_digs_raw.mean(axis=0), axis=1).mean():.3f} m")
print(f"  Min/Max dist: {np.linalg.norm(all_digs_raw - all_digs_raw.mean(axis=0), axis=1).min():.3f} - {np.linalg.norm(all_digs_raw - all_digs_raw.mean(axis=0), axis=1).max():.3f} m")

head_center = all_digs_raw.mean(axis=0)
digs_centered = all_digs_raw - head_center
symmetry_check = np.corrcoef(digs_centered[:, 0], -digs_centered[:, 0])[0, 1]
print(f"  L-R symmetry: {symmetry_check:.3f} (should be > 0.9)")


In [ ]:
epochs = mne.read_epochs(str(RAW_FILE_PATH), preload=False, verbose="ERROR")


print(f"✓ Epochs data loaded:")
print(f"  - Trials: {len(epochs)}")
print(f"  - Channels: {epochs.info['nchan']}")
print(f"  - Sampling rate: {epochs.info['sfreq']:.1f} Hz")
print(f"  - Digitized points: {len(epochs.info['dig'])}")

print(f"\n[STEP 2.1] Загрузка BEM-модели...")

bem_surfs = mne.read_bem_solution(str(bem_path))
print(f"✓ BEM loaded with {len(bem_surfs)} surface(s)\n")

In [ ]:
if freesurfer_dir is None or not (freesurfer_dir / "bem").exists():
    print(f" freesurfer_dir is invalid or 'bem' directory not found!")
    print(f"   freesurfer_dir = {freesurfer_dir}")
    print(f"   SUBJECTS_DIR = {SUBJECTS_DIR}")
    raise RuntimeError(f"Cannot access FreeSurfer BEM directory")

print(f"Co-registration MEG → {TEMPLATE_NAME}...\n")

print(f"Loading data...")
epochs = mne.read_epochs(str(RAW_FILE_PATH), preload=False, verbose="ERROR")

print(f"✓ Epochs data loaded:")
print(f"  - Trials: {len(epochs)}")
print(f"  - Channels: {epochs.info['nchan']}")
print(f"  - Sampling rate: {epochs.info['sfreq']:.1f} Hz")
print(f"  - Digitized points: {len(epochs.info['dig'])}")

print(f"\n Loading BEM model...")
bem_surfs = mne.read_bem_solution(str(bem_path))
print(f"✓ BEM loaded with {len(bem_surfs['surfs'])} surface(s)\n")

# CREATE outer_skin.surf FROM BEM
outer_skin_path = freesurfer_dir / "bem" / "outer_skin.surf"
print(f" Checking outer_skin.surf...")


In [ ]:

print(f"\n Co-registration via MNE Coregistration API...\n")

from mne.coreg import Coregistration
from scipy.spatial import cKDTree

# MNE Coregistration expects subjects_dir = PARENT directory of the subject folder
subjects_dir_coreg = str(SUBJECTS_DIR.parent if (SUBJECTS_DIR / 'bem').exists() else SUBJECTS_DIR)
print(f"  subjects_dir for coreg: {subjects_dir_coreg}")
print(f"  subject:                {TEMPLATE_NAME}")

# PRE-CHECK: check and fix fiducial ident codes
# FIFF standard: ident=1 = LPA, ident=2 = Nasion, ident=3 = RPA
# In MEG head frame: Nasion → max Y, LPA → min X, RPA → max X
print("\n[4.0] Checking fiducial ident codes (FIFF: 1=LPA, 2=Nasion, 3=RPA)...")

_meg_fids = {d['ident']: d['r'].copy() for d in epochs.info['dig'] if d['kind'] == 1}
_names_fiff = {1: 'LPA', 2: 'Nasion', 3: 'RPA'}
for ident, name in [(1, 'LPA'), (2, 'Nasion'), (3, 'RPA')]:
    pt = _meg_fids.get(ident)
    if pt is not None:
        print(f"    ident={ident} ({name:6s}): x={pt[0]*1000:6.1f}  y={pt[1]*1000:6.1f}  z={pt[2]*1000:6.1f} mm")

_y_vals   = {i: pt[1] for i, pt in _meg_fids.items()}
_x_vals   = {i: pt[0] for i, pt in _meg_fids.items()}
_true_nas = max(_y_vals, key=_y_vals.get)               # max Y → Nasion
_rest     = [i for i in _meg_fids if i != _true_nas]
_true_lpa = min(_rest, key=lambda i: _x_vals[i])        # min X → LPA
_true_rpa = max(_rest, key=lambda i: _x_vals[i])        # max X → RPA

# FIFF: Nasion=2, LPA=1, RPA=3
_fix_map = {_true_nas: 2, _true_lpa: 1, _true_rpa: 3}  # current_ident → correct_fiff_ident

if any(k != v for k, v in _fix_map.items()):
    print(f"\n Wrong ident codes detected! Fixing:")
    for orig, correct in _fix_map.items():
        if orig != correct:
            print(f"    ident={orig} ({_names_fiff.get(orig,'?')}) → actually {_names_fiff.get(correct,'?')} (ident={correct})")

    _info_coreg = epochs.info.copy()
    def _apply_remap():
        for d in _info_coreg['dig']:
            if d['kind'] == 1 and d['ident'] in _fix_map:
                d['ident'] = _fix_map[d['ident']]
    try:
        _apply_remap()
    except Exception:
        try:
            with _info_coreg._unlock():
                _apply_remap()
        except Exception as _exc:
            print(f" Could not fix: {_exc} — using original")
            _info_coreg = epochs.info

    # Verify fix
    _fixed_fids = {d['ident']: d['r'] for d in _info_coreg['dig'] if d['kind'] == 1}
    print(f"  ✓ Ident codes fixed:")
    for ident, name in [(1, 'LPA'), (2, 'Nasion'), (3, 'RPA')]:
        pt = _fixed_fids.get(ident)
        if pt is not None:
            print(f"    ident={ident} ({name:6s}): x={pt[0]*1000:6.1f}  y={pt[1]*1000:6.1f}  z={pt[2]*1000:6.1f} mm")
else:
    print(f"  ✓ Ident codes correct (LPA=ident{_true_lpa}, Nasion=ident{_true_nas}, RPA=ident{_true_rpa})")
    _info_coreg = epochs.info

#  Fiducial alignment
print("\n[4.1] Fiducial alignment...")
coreg = Coregistration(_info_coreg, subject=TEMPLATE_NAME, subjects_dir=subjects_dir_coreg)
coreg.fit_fiducials(verbose=False)

dist_fid = coreg.compute_dig_mri_distances()
print(f"  Error after fiducial alignment: mean={dist_fid.mean()*1000:.2f} mm  max={dist_fid.max()*1000:.2f} mm")

# ICP refinement
print("\n[4.2] ICP refinement...")
coreg.fit_icp(n_iterations=40, verbose=False)

dist_icp = coreg.compute_dig_mri_distances()
print(f"  Error after ICP: mean={dist_icp.mean()*1000:.2f} mm  max={dist_icp.max()*1000:.2f} mm")

# Get trans
trans        = coreg.trans
trans_matrix = trans['trans'].copy()

print(f"\n  Trans: frame {trans['from']} → frame {trans['to']}")
print(f"  Top 3×4 matrix:\n{trans_matrix[:3]}")

# Validate: where do physical fiducials land in MRI space
# Take positions from original info (in meters, head frame)
_nasion_pt = _meg_fids.get(_true_nas)  # physical Nasion (max Y)
_lpa_pt    = _meg_fids.get(_true_lpa)  # physical LPA (min X)
_rpa_pt    = _meg_fids.get(_true_rpa)  # physical RPA (max X)

if _nasion_pt is not None:
    def _to_mri(pt):
        return (trans_matrix @ np.append(pt, 1))[:3] * 1000
    nas_mri = _to_mri(_nasion_pt)
    lpa_mri = _to_mri(_lpa_pt)
    rpa_mri = _to_mri(_rpa_pt)
    print(f"\n  Fiducial positions in MRI (mm):")
    print(f"    Nasion → y={nas_mri[1]:.1f}  (should be > 0, anterior)")
    print(f"    LPA    → x={lpa_mri[0]:.1f}  (should be < 0, left)")
    print(f"    RPA    → x={rpa_mri[0]:.1f}  (should be > 0, right)")
    if nas_mri[1] > 0 and lpa_mri[0] < 0 and rpa_mri[0] > 0:
        print(f"  ✓ Orientation correct!")
    else:
        print(f" Unexpected orientation → run GUI coreg (cell below)")

# Prepare variables for subsequent cells
bem_data_auto   = mne.read_bem_solution(str(bem_path))
head_mesh_auto  = bem_data_auto['surfs'][0]
vertices_auto   = head_mesh_auto['rr']
dig_points_auto = np.array([d['r'] for d in epochs.info['dig'] if d['kind'] == 4])

# Plotly visualization
print("\n[4.3] Co-registration visualization (Plotly)...")
try:
    import plotly.graph_objects as go

    dig_4d        = np.hstack([dig_points_auto, np.ones((len(dig_points_auto), 1))])
    dig_in_mri    = (trans_matrix @ dig_4d.T).T[:, :3]
    dig_in_mri_mm = dig_in_mri * 1000

    _tree = cKDTree(vertices_auto)
    dig_errors, _ = _tree.query(dig_in_mri)
    error_mm      = dig_errors * 1000
    head_verts_mm = vertices_auto * 1000

    fig = go.Figure()
    fig.add_trace(go.Mesh3d(
        x=head_verts_mm[:, 0], y=head_verts_mm[:, 1], z=head_verts_mm[:, 2],
        i=head_mesh_auto['tris'][:, 0],
        j=head_mesh_auto['tris'][:, 1],
        k=head_mesh_auto['tris'][:, 2],
        color='lightblue', opacity=0.3, name='BEM Head', hoverinfo='skip'
    ))
    fig.add_trace(go.Scatter3d(
        x=dig_in_mri_mm[:, 0], y=dig_in_mri_mm[:, 1], z=dig_in_mri_mm[:, 2],
        mode='markers',
        marker=dict(size=4, color=error_mm, colorscale='RdYlGn_r',
                    cmin=0, cmax=15, colorbar=dict(title="Error (mm)"), showscale=True),
        text=[f"Err: {e:.1f} mm" for e in error_mm],
        hoverinfo='text', name='Digitized Points'
    ))
    if _nasion_pt is not None:
        _fid_pts = np.array([_nasion_pt, _lpa_pt, _rpa_pt])
        _fid_4d  = np.hstack([_fid_pts, np.ones((3, 1))])
        _fid_mri = (trans_matrix @ _fid_4d.T).T[:, :3] * 1000
        fig.add_trace(go.Scatter3d(
            x=_fid_mri[:, 0], y=_fid_mri[:, 1], z=_fid_mri[:, 2],
            mode='markers+text',
            marker=dict(size=10, color=['red', 'blue', 'green'], symbol='diamond'),
            text=['Nasion', 'LPA', 'RPA'], textposition='top center',
            name='Fiducials (MEG→MRI)', hoverinfo='text'
        ))

    _up   = dict(x=0, y=0, z=1)
    _cams = {
        'front':     dict(x=0,    y=2.5,  z=0  ),
        'back':      dict(x=0,    y=-2.5, z=0  ),
        'left':      dict(x=-2.5, y=0,    z=0  ),
        'right':     dict(x=2.5,  y=0,    z=0  ),
        'top':       dict(x=0,    y=0,    z=2.5),
        'isometric': dict(x=1.5,  y=1.5,  z=1.2),
    }
    fig.update_layout(
        title=(f"<b>Co-registration (MNE API)</b><br>"
               f"<sub>Mean: {error_mm.mean():.1f} mm | Max: {error_mm.max():.1f} mm</sub>"),
        scene=dict(
            xaxis_title='X → right (mm)', yaxis_title='Y → anterior (mm)',
            zaxis_title='Z → superior (mm)', aspectmode='data',
            camera=dict(eye=_cams['front'], up=_up),
        ),
        updatemenus=[dict(
            type='buttons', direction='right', x=0.0, y=1.08, showactive=True,
            buttons=[dict(label=l, method='relayout',
                         args=[{'scene.camera': dict(eye=e, up=_up)}])
                     for l, e in _cams.items()]
        )],
        width=1200, height=800,
    )
    coreg_viz_path = OUTPUT_DIR / "coregistration_result.html"
    fig.write_html(str(coreg_viz_path))
    print(f"  ✓ Plot saved: {coreg_viz_path}")
    fig.show()

except Exception as e:
    print(f" Plotly error: {e}")
    import traceback; traceback.print_exc()

print(f"\n✓ Co-registration done!")
print(f"  Mean error: {dist_icp.mean()*1000:.2f} mm")
print(f"  For manual adjustment → cell below (MNE GUI coreg)")


In [ ]:
# [OPTIONAL] Remove spurious digitizer points at the front of the head
# Two methods — choose one:

import numpy as np

# method A: Y-coordinate threshold
# Everything ANTERIOR to nasion (Y > nasion_Y + margin_mm) is an outlier
_nasion_y_m   = _nasion_pt[1]                    # nasion Y in meters (head frame)
_margin_m     = 0.010                            # 10 mm tolerance, can be reduced
_y_thresh     = _nasion_y_m + _margin_m

#  method B: distance from BEM surface
from scipy.spatial import cKDTree as _KDTree
_tree_bem = _KDTree(vertices_auto)               # vertices_auto — BEM vertices in meters
_dists_bem, _ = _tree_bem.query(dig_points_auto) # in meters
_dist_thresh_m = 0.015                           # discard points > 15 mm from scalp

# Combine: point is spurious if it is ANTERIOR to nasion AND far from BEM
_mask_keep = ~(
    (dig_points_auto[:, 1] > _y_thresh) &       # too far anterior OR
    (_dists_bem > _dist_thresh_m)               # too far from scalp
)

n_before = len(dig_points_auto)
dig_points_filtered = dig_points_auto[_mask_keep]
n_removed = n_before - len(dig_points_filtered)
print(f"[FILTER] Removed points: {n_removed}/{n_before}  →  remaining {len(dig_points_filtered)}")

# Apply - overwrite variable for visualization and coreg
dig_points_auto = dig_points_filtered

# Remove same points from epochs.info['dig'] (for passing to coreg)
_keep_idx = np.where(_mask_keep)[0]
_dig_kind4 = [d for d in epochs.info['dig'] if d['kind'] == 4]
_dig_other  = [d for d in epochs.info['dig'] if d['kind'] != 4]
_dig_kept4  = [_dig_kind4[i] for i in _keep_idx]

try:
    epochs.info['dig'] = _dig_other + _dig_kept4
except Exception:
    with epochs.info._unlock():
        epochs.info['dig'] = _dig_other + _dig_kept4

print(f"epochs.info['dig'] updated: {len(_dig_kind4)} → {len(_dig_kept4)} kind=4 points")
print("  Re-run the co-registration cell to recompute ICP without spurious points.")

# Quick check — Y distribution of remaining points (in mm)
_y_mm = dig_points_auto[:, 1] * 1000
print(f"  Remaining Y range: min={_y_mm.min():.1f}  max={_y_mm.max():.1f}  nasion={_nasion_y_m*1000:.1f} mm")


In [ ]:
print('\n Launching MNE coreg GUI...')
print(f'  Save trans to: {trans_out}')

# subjects_dir for GUI should be the parent directory that contains the subject folder
subjects_dir_gui = SUBJECTS_DIR.parent if (SUBJECTS_DIR / 'bem').exists() else SUBJECTS_DIR

# Use a file path for inst (this MNE version expects a path)
inst_path = RAW_FILE_PATH if RAW_FILE_PATH.exists() else None
if inst_path is None and hasattr(raw, 'filenames') and raw.filenames:
    inst_path = raw.filenames[0]
if inst_path is None:
    raise RuntimeError('Set RAW_FILE_PATH to a valid .fif for coreg GUI')

mne.gui.coregistration(
    subject=TEMPLATE_NAME,
    subjects_dir=str(subjects_dir_gui),
    inst=str(inst_path),
)

# After saving in the GUI, load the trans file here
if trans_out.exists():
    trans = mne.read_trans(trans_out)
    print(f'Loaded trans: {trans_out}')
else:
    print('trans_out not found yet. Save it from the GUI and re-run this cell.')


In [ ]:
# LOAD MANUAL TRANS (after saving from GUI)
# Run this cell if you already saved a trans from the GUI and want to continue without auto-ICP.

if trans_out.exists():
    trans = mne.read_trans(str(trans_out))
    print(f" Loaded trans from file: {trans_out}")
    print(f"  Type: {type(trans)}")
    print(f"  from={trans['from']}  to={trans['to']}")
else:
    print(f" File not found: {trans_out}")
    print(f"   Make sure you saved the trans in the GUI to this path:")
    print(f"   {trans_out}")


In [ ]:
print(f"Visualizing alignment result on fsaverage...\n")

# BEM path already set in config
bem_path_viz = bem_path

try:
    fig = mne.viz.plot_alignment(
        info=epochs.info,
        trans=trans,
        subject=TEMPLATE_NAME,
        subjects_dir=str(SUBJECTS_DIR),
        dig=True,  # show digitized points
        bem=str(bem_path_viz),
        surfaces=['head'],  # head surface only
        show_axes=True,
        coord_frame='mri',
    )
    print("\n Plot done (close window to continue)\n")

except Exception as e:
    print(f"\nFull visualization unavailable: {e}")
    print("  The transformation is already computed and will be used for analysis.\n")


In [ ]:
print(f"Saving trans file to {OUTPUT_DIR.name}...\n")

# Create output directory if needed
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Save transformation
mne.write_trans(
    str(trans_out),
    trans,
    overwrite=True
)

print(f"✓ Saved: {trans_out}")
print(f"\nFile saved to: {OUTPUT_DIR.name}")
print(f"\nOutput structure:")
print(f"  derivatives/")
print(f"  ├─ {OUTPUT_DIR.name}/")
print(f"  │  ├─ {SUBJECT}-trans.fif ✓ (transformation matrix)")
print(f"  │  └─ {SUBJECT}_oct6-inv.fif (after computing inverse)")
print(f"  ├─ epochs_combined_S.fif")
print(f"  ├─ noise_cov_baseline_S.fif")
print(f"  └─ ...")
print("\n" + "="*70)
print("CO-REGISTRATION DONE")
print("="*70)


In [ ]:
print(f"\nLoading/creating source space...\n")

# Load or create source space
src_path_ico = src_path

if src_path_ico.exists():
    # Load existing source space
    src = mne.read_source_spaces(str(src_path_ico), verbose=False)
    print(f"✓ Source space loaded from: {src_path_ico.name}")

    old_subject = src[0].get('subject_his_id', 'NOT SET')
    if old_subject != TEMPLATE_NAME:
        print(f"  Source space subject '{old_subject}' → '{TEMPLATE_NAME}'")
        for src_part in src:
            src_part['subject_his_id'] = TEMPLATE_NAME
        print(f"  ✓ Subject updated to '{TEMPLATE_NAME}'")
else:
    # Create source space (may take a while)
    print(f"  → Building source space (may take a while)...")

    # NOTE: use FREESURFER_NAME for the folder name (not TEMPLATE_NAME!)
    # TEMPLATE_NAME may differ from the actual folder (e.g. 12ryzhik vs Ryzhik)
    if USE_TEMPLATE:
        subject_name = TEMPLATE_NAME
        subject_dir = str(SUBJECTS_DIR)
    else:
        subject_name = FREESURFER_NAME
        subject_dir = str(freesurfer_dir.parent)

    src = mne.setup_source_space(
        subject=subject_name,
        subjects_dir=subject_dir,
        add_dist=False,
        verbose=False
    )

print(f"✓ Source space ready:")
print(f"  - Hemispheres: {len(src)}")
print(f"  - Left hemisphere: {src[0]['nuse']} dipoles")
print(f"  - Right hemisphere: {src[1]['nuse']} dipoles")
print(f"  - Total: {src[0]['nuse'] + src[1]['nuse']} dipoles\n")


In [ ]:

print(f"\n[STEP 7.5] CO-REGISTRATION QUALITY CHECK\n")
print("="*70)

from scipy.spatial import cKDTree

# Thresholds differ by mode: template gives larger errors — that is expected
if USE_TEMPLATE:
    THRESH_MEAN_WARN  = 8.0   # mm — warning
    THRESH_MEAN_FAIL  = 12.0  # mm — fail
    THRESH_OUTLIER_MM = 15.0  # mm — outlier
    THRESH_OUTLIER_PC = 0.20  # outlier fraction for fail
    THRESH_STD        = 10.0  # mm
    mode_label        = "template (fsaverage)"
else:
    THRESH_MEAN_WARN  = 5.0
    THRESH_MEAN_FAIL  = 8.0
    THRESH_OUTLIER_MM = 10.0
    THRESH_OUTLIER_PC = 0.10
    THRESH_STD        = 8.0
    mode_label        = "individual MRI"

print(f"  Mode: {mode_label}")
print(f"  Thresholds: mean_warn={THRESH_MEAN_WARN} mm, outlier>{THRESH_OUTLIER_MM} mm (<{THRESH_OUTLIER_PC*100:.0f}%)")

# Load BEM for checking
bem_check   = mne.read_bem_solution(str(bem_path))
head_surface = bem_check['surfs'][0]
head_vertices = head_surface['rr']  # in meters

# Transform EXTRA digitizers (kind=4) to MRI coordinates
dig_points_extra = np.array([d['r'] for d in epochs.info['dig'] if d['kind'] == 4])

if len(dig_points_extra) > 0 and 'trans' in locals():
    dig_4d = np.hstack([dig_points_extra, np.ones((len(dig_points_extra), 1))])
    dig_transformed = (trans['trans'] @ dig_4d.T).T[:, :3]

    tree = cKDTree(head_vertices)
    distances_m, indices = tree.query(dig_transformed)
    distances_mm = distances_m * 1000

    head_center   = head_vertices.mean(axis=0)
    dist_to_center_dig  = np.linalg.norm(dig_transformed - head_center, axis=1)
    dist_to_center_surf = np.linalg.norm(head_vertices[indices] - head_center, axis=1)
    n_inside      = int(np.sum(dist_to_center_dig < dist_to_center_surf - 0.002))  # > 2mm inside

    outliers_far  = int(np.sum(distances_mm > THRESH_OUTLIER_MM))

    print(f"\nSENSOR vs SCALP STATS:")
    print(f"  Total EXTRA points (digitizers): {len(dig_points_extra)}")
    print(f"  Inside scalp (estimate):         {n_inside}")
    print(f"  Farther than {THRESH_OUTLIER_MM:.0f} mm from surface: {outliers_far}  ({outliers_far/len(dig_points_extra)*100:.1f}%)")
    print(f"\n  DISTANCES TO SURFACE:")
    print(f"    Mean:   {distances_mm.mean():.2f} mm")
    print(f"    Median: {np.median(distances_mm):.2f} mm")
    print(f"    Min:    {distances_mm.min():.2f} mm")
    print(f"    Max:    {distances_mm.max():.2f} mm")
    print(f"    Std:    {distances_mm.std():.2f} mm")

    # QC logic
    print(f"\n🔍 QC CHECKS:")
    warnings_qc = []
    qc_fail  = False

    if n_inside > 5:
        warnings_qc.append(f" {n_inside} sensors inside scalp (>5 — serious)")
        qc_fail = True

    if distances_mm.mean() > THRESH_MEAN_FAIL:
        warnings_qc.append(f" Mean distance {distances_mm.mean():.1f} mm > threshold {THRESH_MEAN_FAIL} mm")
        qc_fail = True
    elif distances_mm.mean() > THRESH_MEAN_WARN:
        warnings_qc.append(f" Mean distance {distances_mm.mean():.1f} mm > {THRESH_MEAN_WARN} mm (warning)")

    if outliers_far > len(dig_points_extra) * THRESH_OUTLIER_PC:
        warnings_qc.append(f" {outliers_far} sensors ({outliers_far/len(dig_points_extra)*100:.1f}%) farther than {THRESH_OUTLIER_MM:.0f} mm")
        if not USE_TEMPLATE:
            qc_fail = True  # for template — warning only

    if distances_mm.std() > THRESH_STD:
        warnings_qc.append(f" High std ({distances_mm.std():.1f} mm) — uneven alignment")

    for w in warnings_qc:
        print(f"  {w}")

    if qc_fail:
        print(f"\n  FAILED — serious co-registration problems")
        print(f"     Recommended: redo in MNE GUI (cell above)")
        on_inside_setting = "warn"   # don't crash, but warn
    elif warnings_qc:
        print(f"\n  MARGINAL — co-registration acceptable but has warnings")
        if USE_TEMPLATE:
            print(f"     For fsaverage template this is normal — continuing")
        on_inside_setting = "warn"
    else:
        print(f"\n PASSED — co-registration quality is good!")
        on_inside_setting = "raise"

    # Save QC metric
    qc_pass = not qc_fail
    coreg_qc = {
        "mode": mode_label,
        "n_digitizers":     int(len(dig_points_extra)),
        "n_inside_scalp":   n_inside,
        "n_outliers_far":   outliers_far,
        "distance_mean_mm": float(distances_mm.mean()),
        "distance_median_mm": float(np.median(distances_mm)),
        "distance_min_mm":  float(distances_mm.min()),
        "distance_max_mm":  float(distances_mm.max()),
        "distance_std_mm":  float(distances_mm.std()),
        "qc_pass":          bool(qc_pass),
        "warnings":         warnings_qc,
        "on_inside_setting": on_inside_setting,
    }
    import json
    coreg_qc_path = OUTPUT_DIR / "coreg_qc.json"
    with open(coreg_qc_path, 'w', encoding='utf-8') as f:
        json.dump(coreg_qc, f, indent=2)
    print(f"\n✓ QC metric saved: {coreg_qc_path.name}")

else:
    print("No EXTRA digitizers or trans — skipping QC")
    on_inside_setting = "warn"

print(f"\n→ Forward solution will use: on_inside='{on_inside_setting}'")
print("="*70 + "\n")


In [ ]:
print(f"\nComputing forward model...\n")

bem_path_fwd = bem_path
bem_fwd = mne.read_bem_solution(str(bem_path_fwd))

# Use dynamic on_inside parameter based on QC check
if 'on_inside_setting' not in locals():
    print("WARNING: co-registration QC was not run!")
    print("   Using on_inside='raise' (strict check)")
    on_inside_setting = "raise"

print(f"→ Using on_inside='{on_inside_setting}'")

try:
    fwd = mne.make_forward_solution(
        info=epochs.info,
        trans=trans,
        src=src,
        bem=bem_fwd,
        eeg=False,
        mindist=5.0,
        on_inside=on_inside_setting,
        n_jobs=1,
        verbose=False
    )
except RuntimeError as e:
    if "sources inside the surface" in str(e) or "outside boundary" in str(e):
        print("\n" + "="*70)
        print("ERROR: sensors are INSIDE the scalp!")
        print("="*70)
        print(f"\nError message: {e}")
        print("\nThis means the co-registration quality is poor.")
        print("="*70)
        raise
    else:
        raise

print(f"✓ Forward model computed:")
print(f"  - Dipoles: {fwd['nsource']}")
print(f"  - Channels: {fwd['info']['nchan']}")
print(f"  - Forward matrix: {fwd['sol']['data'].shape}\n")


In [ ]:
print(f"\n Saving forward solution...\n")

# Per-subject/session BIDS-style name, required by LCMV in notebook 4.2
fwd_mode = "fsaverage" if USE_TEMPLATE else "subjectMRI"
fwd_out_path = OUTPUT_DIR / f"sub-{SUBJECT}_ses-{SESSION}_desc-{fwd_mode}_fwd.fif"

mne.write_forward_solution(str(fwd_out_path), fwd, overwrite=True)
print(f"  ✓ Saved: {fwd_out_path.name}")
print(f"  nsource={fwd['nsource']}  nchan={fwd['info']['nchan']}")
print(f"  → LCMV beamformer (4.2) loads this file\n")


In [ ]:

print(f"\nLoading/computing noise covariance...\n")

noise_cov = None
noise_cov_source = None

# 1. Try loading cached noise covariance from empty room
noise_cov_er_path = DERIVATIVES_BASE / f"sub-{SUBJECT}_ses-{SESSION}_desc-emptyroom_noise-cov.fif"

if noise_cov_er_path.exists():
    noise_cov = mne.read_cov(str(noise_cov_er_path))
    noise_cov_source = 'empty-room'
    print(f"✓ Noise covariance loaded from cache (empty room)")
    print(f"  File: {noise_cov_er_path.name}")

# 2. If no cache — search for raw empty room file in {SESSION}_empty_room/
if noise_cov is None:
    # Dir structure: derivatives/{SESSION}_empty_room/raw_er_pre*.fif or raw_er_sss*.fif
    er_dir = DERIVATIVES_BASE / f"{SESSION}_empty_room"
    er_path_found = None

    if er_dir.exists():
        # Priority: SSS (Maxwell-filtered) > pre (raw)
        for pattern in ["raw_er_sss*.fif", "raw_er_sss.fif",
                        "raw_er_pre*.fif", "raw_er_pre.fif",
                        "*.fif"]:
            candidates = sorted(er_dir.glob(pattern))
            if candidates:
                er_path_found = candidates[0]
                break
        if er_path_found:
            print(f"  Found empty room directory: {er_dir}")
            print(f"  File: {er_path_found.name}")
        else:
            print(f"  ⚠️ Directory {er_dir.name} exists but no .fif files found")
    else:
        print(f"  ⚠️ Empty room directory not found: {er_dir}")

    if er_path_found:
        print(f"\nLoading raw empty room (may take ~1-2 minutes)...")
        try:
            er_raw = mne.io.read_raw_fif(str(er_path_found), preload=False, verbose="ERROR")
            print(f"✓ Empty room loaded: {er_raw.n_times} samples,  {er_raw.info['sfreq']:.0f} Hz")

            # Ensure channels are compatible with main data
            er_raw.pick(mne.pick_types(er_raw.info, meg=True, eeg=False, exclude='bads'))

            print(f"  Computing noise covariance...")
            noise_cov = mne.compute_raw_covariance(
                er_raw,
                tmin=0,
                tmax=None,
                method='empirical',
                rank='info',
                verbose=False
            )
            noise_cov_source = 'empty-room'
            print(f"✓ Noise covariance computed from empty room")

            # Cache to disk
            mne.write_cov(str(noise_cov_er_path), noise_cov, overwrite=True)
            print(f"  Cached: {noise_cov_er_path.name}")

        except Exception as e:
            print(f"Empty room processing error: {e}")
            print(f"   Falling back to baseline...")

# 3. Fallback: noise covariance from baseline period of epochs
if noise_cov is None:
    noise_cov_baseline_path = DERIVATIVES_BASE / f"sub-{SUBJECT}_ses-{SESSION}_desc-baseline_noise-cov.fif"

    if noise_cov_baseline_path.exists():
        noise_cov = mne.read_cov(str(noise_cov_baseline_path))
        noise_cov_source = 'baseline'
        print(f"✓ Noise covariance loaded from cache (baseline)")
        print(f"  File: {noise_cov_baseline_path.name}")
    else:
        print(f" Empty room not found — computing from baseline epochs")
        print(f"   (includes physiological noise — less optimal)")
        noise_cov = mne.compute_covariance(
            epochs,
            tmin=None,
            tmax=0,
            method='auto',
            verbose=False
        )
        noise_cov_source = 'baseline'
        print(f"✓ Noise covariance computed from baseline")
        mne.write_cov(str(noise_cov_baseline_path), noise_cov, overwrite=True)
        print(f"  Cached: {noise_cov_baseline_path.name}")


# 4. Logging

print(f"\n  Noise covariance stats:")
print(f"  - Shape:   {noise_cov.data.shape}")
print(f"  - Source:  {noise_cov_source.upper()}")

import json
noise_cov_meta = {
    "source":     noise_cov_source,
    "subject":    SUBJECT,
    "session":    SESSION,
    "n_channels": noise_cov.data.shape[0],
    "shape":      list(noise_cov.data.shape),
    "file_path":  str(noise_cov_er_path if noise_cov_source == 'empty-room' else noise_cov_baseline_path),
}
noise_cov_meta_path = DERIVATIVES_BASE / f"sub-{SUBJECT}_ses-{SESSION}_noise-cov.json"
with open(noise_cov_meta_path, 'w', encoding='utf-8') as f:
    json.dump(noise_cov_meta, f, indent=2, ensure_ascii=False)

print(f"\n✓ Metadata saved: {noise_cov_meta_path.name}")
print(f"  → Source: {noise_cov_source}  (cite in Methods!)")


In [ ]:

print(f"\n[STEP 10] Computing inverse operator...\n")

# Regularization (signal-to-noise ratio)
SNR    = 3.0
LOOSE  = 0.2
DEPTH  = 0.8
METHOD = 'dSPM'
lambda2 = 1.0 / SNR ** 2

inv = mne.minimum_norm.make_inverse_operator(
    info=epochs.info,
    forward=fwd,
    noise_cov=noise_cov,
    loose=LOOSE,
    depth=DEPTH,
    fixed=False,
    verbose=False
)

print(f"✓ Inverse operator computed:")
print(f"  - Method:         Minimum Norm (SVD), normalization {METHOD}")
print(f"  - loose:          {LOOSE}")
print(f"  - depth:          {DEPTH}")
print(f"  - SNR:            {SNR}  (λ² = {lambda2:.4f})")
print(f"  - nsource:        {inv['nsource']}")
print(f"  - cov rank:       {inv['info']['nchan']}")
print(f"  - noise_cov from: {noise_cov_source.upper()}")

# Save inverse operator
mne.minimum_norm.write_inverse_operator(str(inv_out), inv, overwrite=True)
print(f"✓ Saved: {inv_out.name}")

# Record parameters in JSON (for reproducibility / Methods)
import json
inv_meta = {
    "subject":          SUBJECT,
    "session":          SESSION,
    "method":           "minimum_norm",
    "normalization":    METHOD,
    "loose":            LOOSE,
    "depth":            DEPTH,
    "snr":              SNR,
    "lambda2":          lambda2,
    "nsource":          int(inv['nsource']),
    "cov_rank":         int(inv['info']['nchan']),
    "noise_cov_source": noise_cov_source,
    "inv_file":         inv_out.name,
    "coreg_qc":         coreg_qc if 'coreg_qc' in dir() else None,
}
inv_meta_path = OUTPUT_DIR / f"sub-{SUBJECT}_ses-{SESSION}_desc-{_desc}_inv-params.json"
with open(inv_meta_path, 'w', encoding='utf-8') as f:
    json.dump(inv_meta, f, indent=2, ensure_ascii=False)
print(f"✓ Parameters saved: {inv_meta_path.name}")


In [ ]:
# Debug: check current subject_his_id values
print("=== DEBUG: Subject ID ===")
print(f"src[0]['subject_his_id'] = '{src[0]['subject_his_id']}'")
print(f"src[1]['subject_his_id'] = '{src[1]['subject_his_id']}'")
print(f"fwd['src'][0]['subject_his_id'] = '{fwd['src'][0]['subject_his_id']}'")
print(f"inv['src'][0]['subject_his_id'] = '{inv['src'][0]['subject_his_id']}'")
print(f"TEMPLATE_NAME = '{TEMPLATE_NAME}'")


In [ ]:
print(f"\n[STEP 11] Applying inverse to epoch data...\n")

print(f"  Averaging {len(epochs)} epochs...")
evoked_avg = epochs.average()

print(f"  Applying inverse with dSPM...")
stc_avg = mne.minimum_norm.apply_inverse(
    evoked_avg,
    inverse_operator=inv,
    lambda2=lambda2,
    method='dSPM',
    verbose=False
)

print(f"\n✓ Source time course (STC) computed:")
print(f"  - STC shape: {stc_avg.data.shape}")
print(f"  - Sources: {stc_avg.data.shape[0]}")
print(f"  - Time points: {stc_avg.data.shape[1]}")
print(f"  - Time: {stc_avg.times[0]:.3f} - {stc_avg.times[-1]:.3f} s")
print(f"  - Time resolution: {1/(stc_avg.times[1]-stc_avg.times[0]):.1f} Hz\n")
print(f"  ROI extraction → run 4.2 Extract ROI timecourses (fsaverage).ipynb")


In [ ]:
import json, datetime

if 'stc_avg_path' not in dir():
    stc_avg_path = OUTPUT_DIR / f"sub-{SUBJECT}_ses-{SESSION}_desc-{FREESURFER_NAME}_avg-src"
if 'trans_out' not in dir():
    trans_out = OUTPUT_DIR / f"sub-{SUBJECT}_ses-{SESSION}_desc-{FREESURFER_NAME}_trans.fif"
if 'inv_out' not in dir():
    inv_out = OUTPUT_DIR / f"sub-{SUBJECT}_ses-{SESSION}_desc-{FREESURFER_NAME}_inv.fif"

#  Load parcellation labels (if not already defined)
if 'labels' not in dir():
    _subjects_dir_parc = str(SUBJECTS_DIR.parent if (SUBJECTS_DIR / 'bem').exists() else SUBJECTS_DIR)
    _all_labels = mne.read_labels_from_annot(
        FREESURFER_NAME, parc=ROI_PARC,
        subjects_dir=_subjects_dir_parc, verbose=False
    )
    if ROI_LABELS_OF_INTEREST:
        labels = [lab for lab in _all_labels if lab.name in ROI_LABELS_OF_INTEREST]
        _missing = set(ROI_LABELS_OF_INTEREST) - {lab.name for lab in labels}
        if _missing:
            print(f" Labels not found in parcellation: {_missing}")
    else:
        labels = _all_labels
    print(f"  ✓ Loaded {len(labels)} labels from {ROI_PARC}")

print(f"\n[STEP 12] Saving inverse results...\n")

stc_avg.save(str(stc_avg_path), overwrite=True)
print(f"✓ Average STC saved: {stc_avg_path}")

print("\n  Activity stats:")
print(f"    - Min: {stc_avg.data.min():.3f}")
print(f"    - Max: {stc_avg.data.max():.3f}")
print(f"    - Mean: {stc_avg.data.mean():.3f}")
print(f"    - Std: {stc_avg.data.std():.3f}")

max_idx = np.unravel_index(np.argmax(stc_avg.data), stc_avg.data.shape)
peak_time = stc_avg.times[max_idx[1]]
peak_value = stc_avg.data[max_idx]
print(f"\n  Activity peak: t={peak_time*1000:.1f} ms, amplitude={peak_value:.3f}")

print("\n  Creating interactive HTML visualization by ROI...")
try:
    import plotly.graph_objects as go
    import plotly.express as px

    roi_tc_avg = mne.extract_label_time_course(
        stc_avg, labels, inv["src"],
        mode="mean_flip", verbose=False
    )  # shape: (n_labels, n_times)

    colors = px.colors.qualitative.Plotly + px.colors.qualitative.Dark24

    print("  1️⃣  Creating ROI timecourses (Plotly)...")
    fig = go.Figure()
    for i, lab in enumerate(labels):
        fig.add_trace(go.Scatter(
            x=stc_avg.times * 1000,
            y=roi_tc_avg[i],
            mode='lines',
            name=lab.name,
            line=dict(width=2, color=colors[i % len(colors)]),
            hovertemplate=f'<b>{lab.name}</b><br>Time: %{{x:.1f}} ms<br>Amplitude: %{{y:.3f}}<extra></extra>'
        ))

    fig.update_layout(
        title=f'<b>ROI Mean Activity — {SUBJECT} / {SESSION}</b>',
        xaxis_title='Time (ms)',
        yaxis_title='Amplitude (dSPM, mean_flip)',
        hovermode='x unified',
        height=600, width=1400,
        template='plotly_white',
        legend=dict(font=dict(size=11))
    )
    html1_path = OUTPUT_DIR / "stc_roi_timecourses.html"
    fig.write_html(str(html1_path))
    print(f"  ✓ Saved: {html1_path.name}")

    print("  2️⃣  Creating ROI heatmap (Plotly)...")
    roi_names = [lab.name for lab in labels]
    fig_heatmap = go.Figure(data=go.Heatmap(
        z=roi_tc_avg,
        x=stc_avg.times * 1000,
        y=roi_names,
        colorscale='RdBu_r',
        zmid=0,
        hovertemplate='<b>%{y}</b><br>Time: %{x:.1f} ms<br>Amplitude: %{z:.3f}<extra></extra>'
    ))
    fig_heatmap.update_layout(
        title=f'<b>ROI Activity Heatmap — {SUBJECT} / {SESSION}</b>',
        xaxis_title='Time (ms)',
        yaxis_title='ROI',
        height=max(400, len(labels) * 22),
        width=1400
    )
    html2_path = OUTPUT_DIR / "stc_roi_heatmap.html"
    fig_heatmap.write_html(str(html2_path))
    print(f"  ✓ Saved: {html2_path.name}")

    print(f"\n  ✓ HTML files saved to {OUTPUT_DIR.name}:")
    print(f"    • stc_roi_timecourses.html — ROI timecourses")
    print(f"    • stc_roi_heatmap.html    — ROI heatmap")

except Exception as e:
    print(f"  ⚠️ Plotly visualization failed: {e}")
    import traceback; traceback.print_exc()

# ── Write processing log ──────────────────────────────────────────────
print("\n  Writing processing_log.json...")
_log_path = OUTPUT_DIR / "processing_log.json"

_existing_log = {}
if _log_path.exists():
    with open(_log_path, "r", encoding="utf-8") as f:
        _existing_log = json.load(f)

_mode = "fsaverage" if FREESURFER_NAME == "fsaverage" else "NativeMRI"

_existing_log["2.2_inverse"] = {
    "timestamp": datetime.datetime.now().isoformat(timespec="seconds"),
    "subject": SUBJECT,
    "session": SESSION,
    "mode": _mode,
    "freesurfer_name": FREESURFER_NAME,
    "params": {
        "method": "dSPM",
        "SNR": float(SNR),
        "lambda2": float(lambda2),
        "roi_parc": ROI_PARC,
        "n_roi": len(labels),
        "roi_labels": [lab.name for lab in labels],
        "n_epochs": int(len(epochs)),
        "sfreq": float(epochs.info["sfreq"]),
    },
    "stc_stats": {
        "min": float(round(stc_avg.data.min(), 4)),
        "max": float(round(stc_avg.data.max(), 4)),
        "mean": float(round(stc_avg.data.mean(), 4)),
        "std": float(round(stc_avg.data.std(), 4)),
        "peak_time_ms": float(round(peak_time * 1000, 1)),
        "peak_amplitude": float(round(peak_value, 4)),
    },
    "output_files": {
        "trans": trans_out.name,
        "inv": inv_out.name,
        "avg_stc_lh": stc_avg_path.stem + "-lh.stc",
        "avg_stc_rh": stc_avg_path.stem + "-rh.stc",
        "html_timecourses": "stc_roi_timecourses.html",
        "html_heatmap": "stc_roi_heatmap.html",
    }
}

with open(_log_path, "w", encoding="utf-8") as f:
    json.dump(_existing_log, f, ensure_ascii=False, indent=2)
print(f"  ✓ Log written: {_log_path.name}")

print("\n✓✓✓ INVERSE SOLUTION DONE ✓✓✓")
print("="*70)
print(f"  • Trans:    {trans_out.name}")
print(f"  • Inverse:  {inv_out.name}")
print(f"  • Avg STC:  {stc_avg_path.name}")
print(f"  • Log:      {_log_path.name}")
print(f"\n  Next step → 4.2 Extract ROI timecourses (fsaverage).ipynb")
